# Portfolio half-hourly forecast — corrected version

Each **Fix n** note refers to `mock_15_solution.md`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/meter_halfhourly_2023.csv.gz")
meters = pd.read_csv("../../data/meters.csv")
print("duplicates:", df.duplicated().sum())

duplicates: 40


**Fix 10 / 11** — drop duplicates; build UTC timestamps from the *local* settlement date.
**Fix 4** — correct the unit fault (M100007, September, Wh) and drop the stuck week, then
put every meter on a complete half-hourly grid so a row shift is a time shift.

In [2]:
df = df.drop_duplicates().copy()
local_midnight = pd.to_datetime(df["settlement_date"]).dt.tz_localize("Europe/London")
df["utc"] = local_midnight.dt.tz_convert("UTC") + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="min")
df["month"] = df["utc"].dt.tz_convert("Europe/London").dt.month

mm = df.groupby(["meter_id", "month"])["kwh"].median().unstack()
bad = mm.div(mm.median(axis=1), axis=0).stack()
bad = bad[bad > 100]
for mid, mon in bad.index:
    sel = (df["meter_id"] == mid) & (df["month"] == mon)
    df.loc[sel, "kwh"] /= 1000
print("unit fixes:", bad.index.tolist())

per_day = df.groupby(["meter_id", "settlement_date"])["kwh"].nunique()
stuck = per_day[per_day <= 1].index
df = df.set_index(["meter_id", "settlement_date"]).drop(stuck).reset_index()
print("stuck meter-days dropped:", len(stuck))

grid = pd.date_range(df["utc"].min(), df["utc"].max(), freq="30min")
wide = df.pivot(index="utc", columns="meter_id", values="kwh").reindex(grid)
wide.index.name = "utc"
print("grid shape:", wide.shape, "| missing cells:", int(wide.isna().sum().sum()))

unit fixes: [('M100007', 9)]


stuck meter-days dropped: 7
grid shape: (17520, 20) | missing cells: 1337


**Fix 1 / 2 / 3** — lags, rolling means and the target are computed *per meter* (on the wide
frame one column is one meter, so a row shift is a time shift for every meter and nothing
crosses a meter boundary). The rolling level must end at the forecast origin t, not be
shifted onto the target day. The target is the value 48 periods ahead.

In [3]:
def stack(frame, name):
    return frame.stack(future_stack=True).rename(name)

feat = pd.concat([
    stack(wide, "kwh"),
    stack(wide.shift(48), "lag48"),
    stack(wide.shift(336), "lag336"),
    stack(wide.rolling(48, min_periods=40).mean(), "roll48"),     # ends at t
    stack(wide.shift(-48), "target"),
    stack(wide.shift(288), "naive_week"),                          # same period last week relative to t+48
], axis=1).reset_index()
local = feat["utc"].dt.tz_convert("Europe/London")
feat["hour"] = local.dt.hour
feat["is_weekend"] = (local.dt.dayofweek >= 5).astype(int)
feat = feat.dropna(subset=["kwh", "lag48", "lag336", "roll48", "target"])
print(feat.shape)

(339655, 10)


**Fix 5** — chronological split: train Jan–Sep, test Oct–Dec.
**Fix 6** — per-meter scaling statistics from the *training* period only.
**Fix 7** — keep all meter dummies (Ridge handles the collinearity), so no meter is a
silent "zero-effect" reference.

In [4]:
split = pd.Timestamp("2023-10-01", tz="Europe/London").tz_convert("UTC")
train = feat[feat["utc"] < split].copy()
test = feat[feat["utc"] >= split].copy()

stats = train.groupby("meter_id")["kwh"].agg(["mean", "std"])
def add_z(d):
    d = d.merge(stats, left_on="meter_id", right_index=True)
    for c in ["kwh", "lag48", "lag336", "roll48", "target"]:
        d[c + "_z"] = (d[c] - d["mean"]) / d["std"]
    return d
train, test = add_z(train), add_z(test)

def design(d):
    return pd.concat([d[["kwh_z", "lag48_z", "lag336_z", "roll48_z", "is_weekend"]],
                      pd.get_dummies(d["hour"], prefix="h").astype(float),
                      pd.get_dummies(d["meter_id"], prefix="m").astype(float)], axis=1)
X_train, X_test = design(train), design(test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)
model = Ridge(alpha=1.0).fit(X_train, train["target_z"])
test["pred_z"] = model.predict(X_test)
test["pred"] = test["pred_z"] * test["std"] + test["mean"]
print("pooled test R2 (kWh):", round(r2_score(test["target"], test["pred"]), 4),
      "| pooled R2 (z-scored):", round(r2_score(test["target_z"], test["pred_z"]), 4))
coef = pd.Series(model.coef_, index=X_train.columns)
coef[["kwh_z", "lag48_z", "lag336_z", "roll48_z", "is_weekend"]].round(4)

pooled test R2 (kWh): 0.796 | pooled R2 (z-scored): 0.4895


kwh_z         0.2240
lag48_z       0.2220
lag336_z      0.2154
roll48_z      0.3032
is_weekend    0.0180
dtype: float64

**Fix 8 / 9** — pooled R² in kWh across meters is inflated by between-meter scale (SME vs
household). Report per-meter R² and MAE, and compare with the naive "same period yesterday"
(the value at the origin t, which is 48 periods before the target) and "same period last
week" per meter.

In [5]:
test["naive_yday"] = test["kwh"]

def per_meter(g):
    gw = g.dropna(subset=["naive_week"])
    return pd.Series({
        "r2_model": r2_score(g["target"], g["pred"]),
        "r2_naive_yday": r2_score(g["target"], g["naive_yday"]),
        "mae_model": mean_absolute_error(g["target"], g["pred"]),
        "mae_naive_yday": mean_absolute_error(g["target"], g["naive_yday"]),
        "mae_naive_week": mean_absolute_error(gw["target"], gw["naive_week"]),
    })
pm = (test.groupby("meter_id")[["target", "pred", "naive_yday", "naive_week"]].apply(per_meter)
        .merge(meters.set_index("meter_id")[["customer_type"]], left_index=True, right_index=True))
pm["model_beats_yday"] = pm["mae_model"] < pm["mae_naive_yday"]
pm["model_beats_week"] = pm["mae_model"] < pm["mae_naive_week"]
print("meters where the model beats same-period-yesterday:", int(pm["model_beats_yday"].sum()), "of", len(pm),
      "| beats same-period-last-week:", int(pm["model_beats_week"].sum()))
print("median per-meter R2: model", round(pm["r2_model"].median(), 3), "| naive yesterday", round(pm["r2_naive_yday"].median(), 3))
pm.round(3).sort_values("r2_model")

meters where the model beats same-period-yesterday: 17 of 20 | beats same-period-last-week: 17
median per-meter R2: model 0.465 | naive yesterday 0.105


,r2_model,r2_naive_yday,mae_model,mae_naive_yday,mae_naive_week,customer_type,model_beats_yday,model_beats_week
meter_id,,,,,,,,
M100017,0.434,0.057,0.063,0.082,0.081,residential,True,True
M100006,0.438,0.060,0.074,0.096,0.095,residential,True,True
M100001,0.444,0.090,0.041,0.053,0.053,residential,True,True
M100019,0.451,0.107,0.063,0.081,0.082,residential,True,True
M100004,0.453,0.055,0.039,0.051,0.051,residential,True,True
M100002,0.454,0.078,0.065,0.084,0.084,residential,True,True
M100003,0.455,0.063,0.047,0.061,0.060,residential,True,True
M100005,0.458,0.102,0.054,0.071,0.071,residential,True,True
M100016,0.459,0.115,0.093,0.120,0.122,residential,True,True


## Honest results

In [6]:
pd.Series({
    "pooled_r2_test_kwh": round(r2_score(test["target"], test["pred"]), 4),
    "pooled_r2_test_z": round(r2_score(test["target_z"], test["pred_z"]), 4),
    "median_per_meter_r2": round(pm["r2_model"].median(), 4),
    "median_per_meter_r2_naive_yday": round(pm["r2_naive_yday"].median(), 4),
    "mean_mae_model": round(pm["mae_model"].mean(), 4),
    "mean_mae_naive_yday": round(pm["mae_naive_yday"].mean(), 4),
    "mean_mae_naive_week": round(pm["mae_naive_week"].mean(), 4),
    "meters_model_beats_naive_yday": int(pm["model_beats_yday"].sum()),
    "n_train": len(X_train), "n_test": len(X_test),
})

pooled_r2_test_kwh                     0.7960
pooled_r2_test_z                       0.4895
median_per_meter_r2                    0.4645
median_per_meter_r2_naive_yday         0.1046
mean_mae_model                         0.1507
mean_mae_naive_yday                    0.1653
mean_mae_naive_week                    0.1618
meters_model_beats_naive_yday         17.0000
n_train                           252527.0000
n_test                             87128.0000
dtype: float64